In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
import torch
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from src.service.finetune.finetune import finetune

In [ ]:
os.makedirs('../_results_with_finetune_with_finetune/finetune', exist_ok=True)
model_names = [
    "resnet50",
    "alexnet",
    "densenet121",
    "mobilenet_v3_small",
    "vgg16",
]

In [ ]:
def save_figure(x, y, xlabel, ylabel, title, model_name, filename):
    fig, ax = plt.subplots()
    ax = sns.lineplot(x=x, y=y, ax=ax)
    ax.set(xlabel=xlabel, ylabel=ylabel, title=title)
    ax.figure.savefig(f"../_results_with_finetune/finetune/{model_name}/{filename}.png")

In [ ]:
for index, model_name in tqdm(enumerate(model_names), position=0, leave=True):
    os.makedirs(f'_results_with_finetune/finetune/{model_name}', exist_ok=True)
    try:
        model = torch.hub.load("pytorch/vision:v0.13.1", model_name, weights="IMAGENET1K_V2")
    except (ValueError, KeyError):
        model = torch.hub.load("pytorch/vision:v0.13.1", model_name, weights="IMAGENET1K_V1")
    #due to limited gpu memory for densenet model
    if model_name == "densenet121":
        batch_size = 32
        num_epochs = 1
    else:
        batch_size = 64
        num_epochs = 3
    train_acc_history, train_loss_history = finetune(model, batch_size=batch_size, num_epochs=num_epochs, feature_extract=False)
    with open(f'../_results_with_finetune/finetune/{model_name}/results.txt', 'w') as f:
        for index, (loss, accuracy) in enumerate(zip(train_loss_history, train_acc_history)):
            f.write(f'Epoch: {index + 1}, Loss: {loss}, Accuracy: {accuracy}\n')
    save_figure(x=range(len(train_acc_history)), y=train_acc_history,
                xlabel="epoch", ylabel="accuracy", title=f"{model_name} accuracy data", 
                model_name=model_name, filename="accuracy")
    save_figure(x=range(len(train_loss_history)), y=train_loss_history,
                xlabel="epoch", ylabel="loss", title=f"{model_name} loss data", 
                model_name=model_name, filename="loss")
    model = model.cpu()
    torch.onnx.export(model, torch.ones(1, 3, 224, 224), f'_results_with_finetune/finetune/{model_name}/model_ft.onnx')
    del model
    torch.cuda.empty_cache()